In [5]:
# Cell 1: Setup & install
# Install ddgs (v9+) — the old "duckduckgo-search" (v8) returns 0 results.
import sys, subprocess, os
try:
    from ddgs import DDGS
    print("ddgs already installed")
except ImportError:
    print("Installing ddgs...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ddgs"])
    from ddgs import DDGS
    print("Install complete")

# 3. Verify we imported the right module
mod = DDGS.__module__
assert "duckduckgo_search" not in mod, f"Wrong module: {mod} — old package conflict"
print(f"DDGS source: {mod} (expected: ddgs)")

# Standard library
import time
import asyncio
from datetime import datetime

# Our data model — ensure project root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from src.search.models import RawCandidate

print("Setup complete ✓")

ddgs already installed
DDGS source: ddgs.ddgs (expected: ddgs)
Setup complete ✓


In [10]:
# Cell 2: Basic feasibility — single query
# Can we get any results? What shape are they? Does the Tesco URL appear?

ddgs = DDGS()
query = "Magic Rock Saucery 4 X 330ML Tesco"

t0 = time.perf_counter()
results = list(ddgs.text(query, max_results=10, region="en"))
elapsed = time.perf_counter() - t0

print(f"Query: '{query}'")
print(f"Results: {len(results)} (took {elapsed:.2f}s)\n")

for i, r in enumerate(results):
    print(f"[{i}] title: {r.get('title', 'N/A')[:120]}")
    print(f"    href:  {r.get('href', 'N/A')[:120]}")
    print(f"    body:  {r.get('body', 'N/A')[:120]}")
    print()

# Map to RawCandidate
candidates = [
    RawCandidate(title=r["title"], url=r["href"], snippet=r.get("body", ""))
    for r in results
]
print(f"Mapped to {len(candidates)} RawCandidate objects")
print(f"First candidate URL: {candidates[0].url if candidates else 'N/A'}")

# Quick check: any tesco.com URLs?
tesco_hits = [c for c in candidates if "tesco.com" in c.url.lower()]
print(f"Tesco URLs found: {len(tesco_hits)}")

Query: 'Magic Rock Saucery 4 X 330ML Tesco'
Results: 10 (took 4.51s)

[0] title: Magic Rock Saucery Session Ipa 330Ml Can - Tesco Groceries
    href:  https://www.tesco.com/shop/en-GB/products/303350818
    body:  If you have any queries, or you'd like advice on any Tesco brand products, please contact Tesco Customer Services, or th

[1] title: Magic Rock Saucery Session IPA 330ml - Compare Prices UK
    href:  https://bargainsforthehome.co.uk/product/magic-rock-saucery-session-ipa-330ml/
    body:  Magic Rock Saucery Session IPA 330ml - Compare supermarket prices to find the best price SAVE money today!

[2] title: Magic Rock Saucery Session IPA 3.9% (330ml) - Compare Prices ...
    href:  https://www.trolley.co.uk/product/magic-rock-saucery-session-ipa-3-9/UTV923
    body:  Compare the Best Prices and Offers on Magic Rock Saucery Session IPA 3.9% (330ml). Prices updated daily.

[3] title: Saucery - Magic Rock Brewing - Untappd
    href:  https://untappd.com/b/magic-rock-brewing-sauce

In [3]:
# Cell 3: Format compatibility — multiple varied queries
# Test across different categories and retailers.

test_queries = [
    ("Heinz Baked Beans 415g Tesco", "grocery — Tesco"),
    ("Samsung Galaxy S24 Ultra Amazon", "electronics — Amazon"),
    ("Bosch Serie 4 Washing Machine Argos", "appliance — Argos"),
    ("Kopparberg Strawberry & Lime 500ml", "alcohol — any retailer"),
    ("LEGO Star Wars Millennium Falcon", "toys — any retailer"),
]

ddgs = DDGS()
all_ok = True

for query, category in test_queries:
    t0 = time.perf_counter()
    try:
        results = list(ddgs.text(query, max_results=5))
    except Exception as e:
        print(f"ERROR '{query}': {type(e).__name__}: {e}")
        all_ok = False
        continue
    elapsed = time.perf_counter() - t0

    domains = set()
    bad = 0
    for r in results:
        href = r.get("href", "")
        if not href:
            bad += 1
            continue
        # Extract domain
        try:
            from urllib.parse import urlparse
            host = urlparse(href).netloc or href
            domains.add(host)
        except Exception:
            pass

    print(f"[{category}] {len(results)} results, {bad} missing hrefs, {elapsed:.2f}s")
    if domains:
        print(f"    domains: {', '.join(sorted(domains)[:5])}")
    if bad:
        all_ok = False
    time.sleep(0.5)  # polite delay

print(f"\nAll queries returned clean results: {all_ok}")

[grocery — Tesco] 5 results, 0 missing hrefs, 1.17s
    domains: www.amazon.co.uk, www.birminghammail.co.uk, www.tesco.com, www.tesco.ie, www.walmart.com
[electronics — Amazon] 5 results, 0 missing hrefs, 1.01s
    domains: www.20minutes.fr, www.amazon.fr, www.ouest-france.fr
[appliance — Argos] 5 results, 0 missing hrefs, 1.24s
    domains: argos-support.co.uk, www.argos.co.uk
[alcohol — any retailer] 5 results, 0 missing hrefs, 1.06s
    domains: kopparberg.com, kopparberg.us, www.ocado.com, www.tesco.com, www.tesco.ie
[toys — any retailer] 5 results, 0 missing hrefs, 0.82s
    domains: www.lego.com, www.reddit.com, www.youtube.com

All queries returned clean results: True


In [8]:
QUERY = "Nescafe Gold Instant Coffee 200g Tesco"
results = list(ddgs.text(QUERY, max_results=5))

In [9]:
results

[{'title': 'Nescafé Gold Blend Intense 200g - PriceRunner',
  'href': 'https://www.pricerunner.com/pl/620-3202859044/Food-Drinks/Nescafe-Gold-Blend-Intense-200g-Compare-Prices',
  'body': 'In stockOffers a rich and full-bodied taste · Features freeze-dried instant coffee · Designed for quick preparation · Suitable for various serving sizes · Comes in a ...'},
 {'title': '[PDF] Aldi Price match - Tesco',
  'href': 'https://digitalcontent.api.tesco.com/v2/media/ghs-roi/fd24ea55-0293-4dd0-ba8b-e8607cd93f95/Aldi_Price_Match_25+Feb.pdf',
  'body': '23 Feb 2026 · NESCAFE GOLD BLEND INSTANT COFFEE 200G. 200G. 8.75 NESCAFE GOLD BLEND INSTANT COFFEE 200G. 200G. 8.75. 8.75. TESCO FREE FROM FUSILLI 500G. 500G.'},
 {'title': 'Nescafé Gold Rich Caramel & Smooth Vanilla Flavours £4 at Tesco…',
  'href': 'https://www.facebook.com/couponmumuk/posts/nescafé-gold-rich-caramel-smooth-vanilla-flavours-4-at-tesco/1115831653920745/',
  'body': '30 Apr 2025 · Coming soon! *NEW* Nescafé Gold Blend flavoured c

In [7]:
# Cell 4: Rate limiting / stability test
# Run consecutive queries at different delay intervals to find the safe call rate.

QUERY = "Nescafe Gold Instant Coffee 200g Tesco"
N_CALLS = 15

def test_rate(delay_s: float, n: int = N_CALLS) -> dict:
    """Run n queries with given delay, return stats."""
    ddgs = DDGS()
    times = []
    errors = 0
    result_counts = []

    for i in range(n):
        if delay_s > 0:
            time.sleep(delay_s)
        t0 = time.perf_counter()
        try:
            results = list(ddgs.text(QUERY, max_results=5))
            elapsed = time.perf_counter() - t0
            times.append(elapsed)
            result_counts.append(len(results))
        except Exception as e:
            elapsed = time.perf_counter() - t0
            times.append(elapsed)
            result_counts.append(0)
            errors += 1
            err_msg = str(e)[:100]
            print(f"  call {i+1} ERROR ({elapsed:.2f}s): {type(e).__name__}: {err_msg}")
        if i < 3 or i >= n - 1:
            print(f"  call {i+1}: {result_counts[-1]} results in {times[-1]:.2f}s")

    avg_time = sum(times) / len(times) if times else 0
    return {
        "delay": delay_s,
        "errors": errors,
        "avg_time": avg_time,
        "min_time": min(times) if times else 0,
        "max_time": max(times) if times else 0,
        "avg_results": sum(result_counts) / len(result_counts) if result_counts else 0,
    }

# Test with different delays
print("=" * 60)
print("Testing with NO delay (0s) — aggressive")
print("=" * 60)
stats_no_delay = test_rate(0.0, n=N_CALLS)

print()
print("=" * 60)
print("Testing with 1s delay — moderate")
print("=" * 60)
stats_1s = test_rate(1.0, n=N_CALLS)

print()
print("=" * 60)
print("Testing with 2s delay — conservative")
print("=" * 60)
stats_2s = test_rate(2.0, n=N_CALLS)

# Summary
print()
print("=" * 60)
print("SUMMARY")
print("=" * 60)
for s in [stats_no_delay, stats_1s, stats_2s]:
    print(f"  delay={s['delay']:.0f}s: {s['errors']} errors, "
          f"{s['avg_time']:.2f}s avg ({s['min_time']:.2f}-{s['max_time']:.2f}), "
          f"{s['avg_results']:.1f} avg results")

Testing with NO delay (0s) — aggressive
  call 1: 5 results in 1.71s
  call 2: 5 results in 1.43s
  call 3: 5 results in 0.79s
  call 15: 5 results in 0.90s

Testing with 1s delay — moderate
  call 1: 5 results in 2.51s
  call 2: 5 results in 0.46s
  call 3: 5 results in 1.15s
  call 15: 5 results in 0.77s

Testing with 2s delay — conservative
  call 1: 5 results in 1.26s
  call 2: 5 results in 1.08s
  call 3: 5 results in 0.76s
  call 15: 5 results in 0.81s

SUMMARY
  delay=0s: 0 errors, 0.89s avg (0.36-1.71), 5.0 avg results
  delay=1s: 0 errors, 0.98s avg (0.46-2.51), 5.0 avg results
  delay=2s: 0 errors, 0.81s avg (0.37-1.62), 5.0 avg results


In [ ]:
# Cell 5: Country/region parameter test
# DuckDuckGo supports `region` — test if it affects result locality.

ddgs = DDGS()
query = "L'Oréal Paris Revitalift Filler"

# Map our country codes to DuckDuckGo region codes
# DDG uses format like: "uk-en", "de-de", "fr-fr", "us-en"
regions_to_test = {
    "uk-en": "United Kingdom",
    "de-de": "Germany",
    "fr-fr": "France",
    "us-en": "United States",
    "wt-wt": "No region (global)",
}

for region_code, region_label in regions_to_test.items():
    t0 = time.perf_counter()
    try:
        if region_code == "wt-wt":
            results = list(ddgs.text(query, max_results=5))
        else:
            results = list(ddgs.text(query, max_results=5, region=region_code))
        elapsed = time.perf_counter() - t0
    except Exception as e:
        print(f"{region_label} ({region_code}): ERROR — {type(e).__name__}: {str(e)[:120]}")
        continue

    domains = set()
    for r in results:
        try:
            from urllib.parse import urlparse
            host = urlparse(r.get("href", "")).netloc
            if host:
                domains.add(host)
        except Exception:
            pass

    print(f"{region_label} ({region_code}): {len(results)} results in {elapsed:.2f}s")
    if domains:
        print(f"    domains: {', '.join(sorted(domains)[:5])}")
    time.sleep(0.5)

print("\nNote: region support may be limited. 'wt-wt' = no region filter.")

In [ ]:
# Cell 6: Edge cases
# Test queries that might break the search engine.

ddgs = DDGS()

edge_cases = [
    ("AAA", "very short query"),
    ("Kühlschrank mit Gefrierfach Edelstahl 180cm Energieklasse A+++ Siemens", "long German query with umlauts"),
    ("Crème Brûlée à la vanille de Madagascar 500ml", "French accents"),
    ("xyznonexistentproduct12345", "nonsense — likely no results"),
    ("Sony WH-1000XM5 Wireless Noise Cancelling Headphones Black", "very common product — many results"),
]

for query, description in edge_cases:
    t0 = time.perf_counter()
    try:
        results = list(ddgs.text(query, max_results=5))
        elapsed = time.perf_counter() - t0
    except Exception as e:
        print(f"[{description}] ERROR: {type(e).__name__}: {str(e)[:150]}")
        time.sleep(1)
        continue

    bad_titles = sum(1 for r in results if not r.get("title", "").strip())
    bad_hrefs = sum(1 for r in results if not r.get("href", "").strip())
    print(f"[{description}]")
    print(f"    Query: '{query[:80]}{'...' if len(query) > 80 else ''}'")
    print(f"    {len(results)} results, {bad_titles} empty titles, {bad_hrefs} empty hrefs, {elapsed:.2f}s")
    if results:
        print(f"    First: {results[0].get('title', 'N/A')[:100]}")
    time.sleep(0.5)

# Cell 7: Summary & Recommendation

Fill this in **after running cells 1-6** above.

---

## Result quality vs Serper

- _(your assessment here)_
- Are the results relevant? Do marketplace URLs (tesco.com, amazon.de, etc.) appear?
- How does result count compare? (Serper typically returns 10 organic results)

---

## Rate limit findings

- **No delay**: _(errors? avg time?)_
- **1s delay**: _(errors? avg time?)_
- **2s delay**: _(errors? avg time?)_
- **Recommended safe call rate**: _(calls per second)_

---

## Country / region support

- Which regions returned country-specific results?
- Does `region` parameter actually change the result set?

---

## Edge cases

- Short queries: _(handled OK?)_
- Non-Latin characters (ü, é, à): _(handled OK?)_
- No-result queries: _(graceful empty list? error?)_

---

## Overall verdict

- [ ] **Viable** — proceed with implementing `DuckDuckGoProvider`
- [ ] **Not viable** — stick with Serper only

### If viable: implementation notes

- What retry strategy is needed?
- What delay between calls?
- Region code mapping: `country` → DDG `region`?
- Any result fields that need transformation?
- Async wrapper needed? (DDGS is synchronous — use `asyncio.to_thread`)